# 🤖 Week 08 — Machine Learning เบื้องต้น (Scikit-Learn Regression)

โมเดล Machine Learning ไม่ใช่เวทมนตร์ แต่เป็นการเรียนรู้ความสัมพันธ์ทางสถิติระหว่าง **Features ($X$)** (เช่น overall, age, pace) กับ **Target ($y$)** (เช่น market value) เพื่อนำไปทำนายมูลค่านักเตะชุดใหม่ที่ไม่เคยเห็นมาก่อน (Unseen Data)


## 📌 ขั้นตอนการสร้างโมเดล Machine Learning

1. **เตรียมข้อมูล (Data Preparation):** จัดการค่าว่างและแปลงคอลัมน์ราคาให้เป็นตัวเลข
2. **แยก Features ($X$) และ Target ($y$):** $X$ เป็น DataFrame 2 มิติ ส่วน $y$ เป็น Series 1 มิติ
3. **แบ่ง Train/Test Split:** แยกข้อมูลไว้ทดสอบความแม่นยำ (ป้องกันการท่องจำข้อสอบ)
4. **ฝึกสอนโมเดล (`.fit()`):** ให้ `LinearRegression` เรียนรู้ค่าน้ำหนัก (Weights / Coefficients)
5. **ประเมินผล (`.predict()`, `MAE`):** วัดความแม่นยำเทียบกับ Baseline (ค่าเฉลี่ย)
6. **ทำนายข้อมูลใหม่ (Inference):** นำโมเดลไปประเมินราคานักเตะจากไฟล์ `scouted_players.csv`


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# 1. โหลดข้อมูลหลัก
df = pd.read_csv("../data/footballers.csv")

# 2. Clean ข้อมูลราคา (แปลงข้อความ unknown ให้เป็น NaN แล้ว drop แถวที่ไม่มีราคา)
df["value_millions"] = pd.to_numeric(df["value_millions"], errors="coerce")
df_clean = df.dropna(subset=["value_millions", "overall", "age", "pace", "shooting"]).copy()

print(f"จำนวนข้อมูลที่พร้อมฝึกโมเดล: {len(df_clean)} คน")
display(df_clean[["name", "overall", "age", "pace", "shooting", "value_millions"]].head(4))

จำนวนข้อมูลที่พร้อมฝึกโมเดล: 44 คน


,name,overall,age,pace,shooting,value_millions
0,Lionel Messi,90,37.0,87,90,45.0
1,Cristiano Ronaldo,86,39.0,78,89,15.0
2,Kylian Mbappe,91,26.0,97,88,180.0
3,Erling Haaland,91,24.0,89,91,170.0


---
## 1️⃣ การแบ่งข้อมูล Features ($X$), Target ($y$) และ Train/Test Split


In [2]:
# 1.1 กำหนด Features (X เป็น 2D Table) และ Target (y เป็น 1D Vector)
feature_cols = ["overall", "age", "pace", "shooting"]
X = df_clean[feature_cols]
y = df_clean["value_millions"]

# 1.2 แบ่ง Train 80% / Test 20% (ตั้ง random_state เพื่อให้ผลลัพธ์คงที่)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"ขนาดชุดฝึก (Train Set): {X_train.shape[0]} แถว")
print(f"ขนาดชุดทดสอบ (Test Set): {X_test.shape[0]} แถว")

ขนาดชุดฝึก (Train Set): 35 แถว
ขนาดชุดทดสอบ (Test Set): 9 แถว


---
## 2️⃣ การเทรนโมเดล LinearRegression และวัดผลความแม่นยำ (MAE)


In [3]:
# 2.1 สร้าง Baseline Model (การทำนายด้วยค่าเฉลี่ย y_train)
baseline_pred = [y_train.mean()] * len(y_test)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

# 2.2 สร้างและฝึกโมเดล Linear Regression (.fit)
model = LinearRegression()
model.fit(X_train, y_train)

# 2.3 ทำนายผลบน Test Set (.predict) และคำนวณ MAE
y_pred = model.predict(X_test)
model_mae = mean_absolute_error(y_test, y_pred)

print(f"1. ความผิดพลาดของ Baseline (MAE): {baseline_mae:.2f} ล้านยูโร")
print(f"2. ความผิดพลาดของ ML Model (MAE): {model_mae:.2f} ล้านยูโร")
print(f"👉 โมเดลมีความแม่นยำดีกว่า Baseline ถึง {((baseline_mae - model_mae)/baseline_mae)*100:.1f}%")

1. ความผิดพลาดของ Baseline (MAE): 29.81 ล้านยูโร
2. ความผิดพลาดของ ML Model (MAE): 17.66 ล้านยูโร
👉 โมเดลมีความแม่นยำดีกว่า Baseline ถึง 40.8%


---
## 3️⃣ การแปลความหมายของโมเดล (Coefficients) & การทำนายข้อมูลชุดใหม่ (Unseen Data)


In [4]:
# 3.1 ตรวจสอบค่าน้ำหนัก (Feature Importance / Coefficients)
coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient (ล้านยูโรต่อ 1 หน่วย)": model.coef_.round(2)
})
print("=== อิทธิพลของแต่ละทักษะต่อมูลค่านักเตะ ===")
display(coef_df)

# 3.2 นำโมเดลไปทำนายราคานักเตะใหม่จากไฟล์ scouted_players.csv
scouted = pd.read_csv("../data/scouted_players.csv")
scouted["predicted_value_millions"] = model.predict(scouted[feature_cols]).round(1)

print("\n=== ผลการประเมินราคานักเตะดาวรุ่งชุดใหม่ (Unseen Data) ===")
display(scouted[["name", "age", "overall", "pace", "shooting", "predicted_value_millions"]])

=== อิทธิพลของแต่ละทักษะต่อมูลค่านักเตะ ===


,Feature,Coefficient (ล้านยูโรต่อ 1 หน่วย)
0,overall,-1.15
1,age,-7.01
2,pace,0.46
3,shooting,1.55



=== ผลการประเมินราคานักเตะดาวรุ่งชุดใหม่ (Unseen Data) ===


,name,age,overall,pace,shooting,predicted_value_millions
0,Lamine Yamal,17,84,85,78,166.9
1,Endrick,18,77,86,78,168.4
2,Arda Guler,19,78,74,75,150.1
3,Warren Zaire-Emery,18,80,76,68,144.9
4,Kobbie Mainoo,19,79,72,66,134.1
5,Alejandro Garnacho,20,80,86,76,147.8
6,Savinho,20,82,88,74,143.3
7,Evan Ferguson,20,77,75,79,150.9
